In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\Sai\Downloads\TX_Harris_Project\Data\Raw\Harris.csv", nrows=100000,low_memory=False)

In [3]:
print(df.describe())
print(df.isnull().sum())

               acct        yr        str_num  Neighborhood_Code  \
count  1.000000e+05  100000.0  100000.000000      100000.000000   
mean   2.421512e+11    2026.0    3003.423980        6230.168357   
std    1.097501e+11       0.0    4188.370274        2717.060419   
min    1.001000e+10    2026.0       0.000000           0.000000   
25%    1.600000e+11    2026.0     142.000000        5904.020000   
50%    2.510000e+11    2026.0    1310.000000        7112.000000   
75%    3.410000e+11    2026.0    4953.250000        8305.090000   
max    4.020000e+11    2026.0   99999.000000        9998.000000   

       Neighborhood_Grp       yr_impr        nxt_bld        bld_ar  \
count      100000.00000  67532.000000  100000.000000  1.000000e+05   
mean         3803.49517   1962.701534       1.732530  4.102144e+03   
std          6887.56008     32.716562       1.233136  4.057260e+04   
min             0.00000   1840.000000       1.000000  0.000000e+00   
25%             0.00000   1938.000000       1.

In [4]:
print(f"Initial rows: {len(df):,}")
print(f"Initial columns: {len(df.columns)}\n")

Initial rows: 100,000
Initial columns: 71



In [5]:
print("Step 1: Dropping useless columns...")
drop_cols = ['certified_date', 'lgl_3', 'lgl_4', 'lgl_2', 'mail_addr_2', 'mail_country', 'undeliverable']
df = df.drop(columns=drop_cols)
print(f"Dropped {len(drop_cols)} columns")
print(f"Remaining columns: {len(df.columns)}\n")

Step 1: Dropping useless columns...
Dropped 7 columns
Remaining columns: 64



In [7]:
print("Step 2: Removing invalid records...")
initial = len(df)
df = df[df['tot_appr_val'] > 0]
print(f"Removed {initial - len(df):,} properties with $0 value")

Step 2: Removing invalid records...
Removed 16,302 properties with $0 value


In [8]:
initial = len(df)
df = df[df['tot_appr_val'] < 100_000_000]
print(f"Removed {initial - len(df):,} properties with >$100M value")

Removed 36 properties with >$100M value


In [9]:
df = df[(df['bld_ar'] == 0) | ((df['bld_ar'] > 100) & (df['bld_ar'] < 500000))]
print(f"Removed {initial - len(df):,} properties with unreasonable building area")

print(f"Final rows after cleaning: {len(df):,}\n")

Removed 103 properties with unreasonable building area
Final rows after cleaning: 83,595



In [30]:
def classify_property_type(state_class):
    """Classify properties based on property tax codes"""
    state_class = str(state_class).strip().upper()
    
    if state_class in ['F1', 'F2', 'C1', 'C2']:
        return 'Commercial'
    elif state_class in ['X1']:
        return 'Government/Exempt'
    elif state_class in ['A1']:
        return 'Agricultural'
    else:
        return 'Other'
df['property_type_group'] = df['state_class'].apply(classify_property_type)
df['is_commercial'] = (df['property_type_group'] == 'Commercial').astype(int)
df['property_age'] = 2025 - df['yr_impr'].fillna(2025).astype(float)
df['property_age'] = df['property_age'].clip(lower=0)
df['total_area'] = df['bld_ar'].fillna(0) + df['land_ar'].fillna(0)
df['prior_value_per_sqft'] = np.where(
    df['total_area'] > 0,
    df['prior_tot_mkt_val'] / df['total_area'],
    0
)
df['prior_land_value_per_sqft'] = np.where(
    df['land_ar'] > 0,
    df['prior_land_val'] / df['land_ar'],
    0
)
df['value_change'] = df['tot_mkt_val'] - df['prior_tot_mkt_val'].fillna(0)
df['value_change_pct'] = np.where(
    df['prior_tot_mkt_val'] > 0,
    (df['value_change'] / df['prior_tot_mkt_val'] * 100),
    0
)
df['prior_land_val'] = df['prior_land_val'].fillna(0)
df['prior_bld_val'] = df['prior_bld_val'].fillna(0)
df['prior_tot_appr_val'] = df['prior_tot_appr_val'].fillna(0)

In [31]:
model_cols = [
    'acct', 'yr',
    'tot_appr_val',
    'state_class', 'property_type_group', 'bld_ar', 'land_ar', 'acreage', 'property_age', 'total_area',
    'prior_land_val', 'prior_bld_val', 'prior_tot_appr_val', 'tot_mkt_val',
    'Neighborhood_Grp', 'Market_Area_1', 'school_dist',
    'prior_value_per_sqft', 'prior_land_value_per_sqft', 'is_commercial', 'value_change_pct',
    'protested', 'value_status', 'noticed'
]

df_model = df[model_cols].copy()

print(f"Selected {len(model_cols)} columns for modeling\n")

Selected 24 columns for modeling



In [33]:
print("Step 5: Handling missing values...")
for col in df_model.select_dtypes(include=[np.number]).columns:
    missing_pct = (df_model[col].isnull().sum() / len(df_model)) * 100
    if missing_pct > 0:
        df_model[col].fillna(df_model[col].median(), inplace=True)
        print(f"Filled {col}: {missing_pct:.1f}% missing")

print()

Step 5: Handling missing values...



In [34]:
print("Step 6: Final data summary...")
print(f"\nTarget variable (tot_appr_val):")
print(f"  Mean: ${df_model['tot_appr_val'].mean():,.0f}")
print(f"  Median: ${df_model['tot_appr_val'].median():,.0f}")
print(f"  Min: ${df_model['tot_appr_val'].min():,.0f}")
print(f"  Max: ${df_model['tot_appr_val'].max():,.0f}")
print(f"  Std Dev: ${df_model['tot_appr_val'].std():,.0f}")

print(f"\nKey features:")
print(f"  Building area: min={df_model['bld_ar'].min():.0f}, max={df_model['bld_ar'].max():.0f}, mean={df_model['bld_ar'].mean():.0f}")
print(f"  Land area: min={df_model['land_ar'].min():.0f}, max={df_model['land_ar'].max():.0f}, mean={df_model['land_ar'].mean():.0f}")
print(f"  Property age: min={df_model['property_age'].min():.0f}, max={df_model['property_age'].max():.0f}, mean={df_model['property_age'].mean():.0f}")

print(f"\nProperty type distribution:")
print(df_model['state_class'].value_counts().head(10))

print(f"\nNeighborhood distribution:")
print(df_model['Neighborhood_Grp'].value_counts().head(10))
output_path = r"C:\Users\Sai\Downloads\TX_Harris_Project\Data\Processed\harris_county_clean.csv"
df_model.to_csv(output_path, index=False)
print(f"Saved cleaned data to: {output_path}")
print(f"Final shape: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns")

print("\n" + "=" * 80)
print("DATA CLEANING COMPLETE - READY FOR MODELING")
print("=" * 80)

Step 6: Final data summary...

Target variable (tot_appr_val):
  Mean: $558,560
  Median: $243,195
  Min: $1
  Max: $96,761,685
  Std Dev: $1,705,047

Key features:
  Building area: min=0, max=499965, mean=3598
  Land area: min=0, max=38703060, mean=29255
  Property age: min=0, max=185, mean=49

Property type distribution:
state_class
A1    47558
F1    12265
C1     8893
C2     6118
B2     2293
C3     1325
B1     1298
A3     1159
A2      620
J3      467
Name: count, dtype: int64

Neighborhood distribution:
Neighborhood_Grp
0        20997
1420      7310
1419      4778
21027     2923
1901      2806
1542      2535
1541      2055
20014     2046
9022      1973
1540      1632
Name: count, dtype: int64
Saved cleaned data to: C:\Users\Sai\Downloads\TX_Harris_Project\Data\Processed\harris_county_clean.csv
Final shape: 83,595 rows × 24 columns

DATA CLEANING COMPLETE - READY FOR MODELING
